# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [ ]:
## Environment setup

# install openwakeword (full installation to support training)
import os
if not os.path.exists("openwakeword"):
    !git clone https://github.com/dscripka/openwakeword
!pip install --no-deps -e ./openwakeword
!pip install -q "onnxruntime>=1.10.0,<2" "ai-edge-litert>=2.0.2,<3"

# install other dependencies
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install pronouncing==0.2.0
!pip install -q -U datasets
!pip install deep-phonemizer==0.0.19

# Download required models (workaround for Colab)
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget -nc https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -nc https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget -nc https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget -nc https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite


**Note (patched for this project):** the original cell here also installed
`piper-sample-generator`/`piper-phonemize` (for synthetic TTS clip generation) and
`tensorflow-cpu`/`tensorflow_probability`/`onnx_tf` (for optional tflite export).
Both are dropped: piper-phonemize has no Python 3.13 wheel at all (Colab default as
of this notebook), and the tflite export path isn't needed since we only use the
ONNX model. Both the positive clips ("rocky") and the synthetic adversarial negative
clips (decoy words/phrases) that piper would normally have generated are instead
produced locally with Kokoro — see the upload cell below, which replaces the
original synthetic-clip-generation step entirely.

In [ ]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html
# (patched: newer `datasets` returns an AudioDecoder object for the "audio" column,
# not the old {'path':..., 'array':...} dict — use .get_all_samples() instead)

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for i, row in enumerate(tqdm(rir_dataset)):
    samples = row["audio"].get_all_samples()
    audio = samples.data.numpy()
    if audio.ndim == 2:
        audio = audio.mean(axis=0)
    scipy.io.wavfile.write(os.path.join(output_dir, f"rir_{i:05d}.wav"), samples.sample_rate, (audio * 32767).astype(np.int16))


In [ ]:
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)
# (patched: converts the already-local .flac files directly with librosa instead of
# going through datasets.Audio/torchcodec — simpler and sidesteps the AudioDecoder change)

import librosa

if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -nc -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

for path in tqdm(list(Path("audioset/audio").glob("**/*.flac"))):
    audio, sr = librosa.load(str(path), sr=16000, mono=True)
    scipy.io.wavfile.write(os.path.join(output_dir, path.name.replace(".flac", ".wav")), 16000, (audio * 32767).astype(np.int16))

# Free Music Archive dataset (https://github.com/mdeff/fma)
# (patched: same AudioDecoder fix as the MIT RIR cell above — streaming HF dataset,
# so we still need datasets' own decoder, just the current API for it)
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset)

n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training
n_clips = n_hours * 3600 // 30  # this works because the FMA dataset is all 30 second clips
for i in tqdm(range(n_clips)):
    row = next(fma_dataset)
    samples = row["audio"].get_all_samples()
    audio = samples.data.numpy()
    if audio.ndim == 2:
        audio = audio.mean(axis=0)
    if samples.sample_rate != 16000:
        audio = librosa.resample(audio, orig_sr=samples.sample_rate, target_sr=16000)
    scipy.io.wavfile.write(os.path.join(output_dir, f"fma_{i:05d}.wav"), 16000, (audio * 32767).astype(np.int16))


In [ ]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [ ]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

In [ ]:
# Modify values in the config and save a new version

config["target_phrase"] = ["rocky"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = ['./audioset_16k', './fma']  # multiple background datasets are supported
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [ ]:
# Step 1 (patched): upload rocky_clips.zip (built locally with training/generate_rocky_clips.py)
# and unzip it into the positive_train/positive_test/negative_train/negative_test dirs
# this pipeline expects, instead of running train.py --generate_clips (which needs
# piper-phonemize for both the positive TTS clips AND the synthetic adversarial negatives).

from google.colab import files
import zipfile
import shutil

uploaded = files.upload()  # select rocky_clips.zip when prompted
zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(".")  # expects rocky_clips/{positive,negative}_{train,test}/*.wav

for split in ["positive_train", "positive_test", "negative_train", "negative_test"]:
    dest_dir = os.path.join(config["output_dir"], config["model_name"], split)
    os.makedirs(dest_dir, exist_ok=True)
    src_dir = os.path.join("rocky_clips", split)
    for fname in os.listdir(src_dir):
        shutil.copy(os.path.join(src_dir, fname), dest_dir)
    print(f"{split}: {len(os.listdir(dest_dir))} clips")


In [ ]:
# Step 2: Augment the generated clips

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

In [ ]:
# Step 3: Train model

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

**Skipped (patched for this project):** the optional tflite re-export retry cell
was here. It needs `tensorflow`/`onnx_tf`, which this patched notebook doesn't
install (no Python 3.13 wheel for the pinned `tensorflow-cpu==2.8.1`, and not
needed since Rocky only uses the `.onnx` model). `train.py --train_model` already
exports `my_custom_model/rocky.onnx` on its own — that's the file to download.

After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!